In [ ]:
# In[1]:

import json

# Load the task.json file
task_directory = 'rca100/cases/t056'
task_json_path = f'{task_directory}/task.json'

with open(task_json_path, 'r') as file:
    task_data = json.load(file)

# Extract alert context
alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window', {})
alert_start = alert_window.get('start')
alert_end = alert_window.get('end')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_start, alert_end, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "inventory错误次数突增" (inventory error count surge). It occurred between '2026-03-28T20:27:36.369643+08:00' and '2026-03-28T20:31:01+08:00'. The alert entity is an APM operation named 'inventory::/api/v1/inventory/{productId}', with the entity type 'apm.operation' and domain 'apm'. The prompt text requests an analysis of the root cause for this issue.

The original code execution output of IPython Kernel is also provided below for reference:

('inventory错误次数突增', '2026-03-28T20:27:36.369643+08:00', '2026-03-28T20:31:01+08:00', {'entity_id': '162799adab6799326676266a2c96cf3e', 'entity_name': 'inventory::/api/v1/inventory/{productId}', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">inventory错误次数突增<alert_event event_id="59eb0b7c0da5dd7cfc7272aeeeaf64e9" trans_id="uhto56n5h90mn0fbd303p7d526" rule_id="663b6c72-c01c-49e2-b1a7-805ae65925b3" rule_name="inventory错误次数突增" /></code> 帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_path = f'{task_directory}/metrics.parquet'
metrics_df = pd.read_parquet(metrics_path)

# Ensure 'time' and 'value' columns are numeric
metrics_df['time'] = pd.to_numeric(metrics_df['time'], errors='coerce')
metrics_df['value'] = pd.to_numeric(metrics_df['value'], errors='coerce')

# Create a new column for 'component-KPI' classification
metrics_df['component_KPI'] = metrics_df['entity_name'] + '-' + metrics_df['metric']

# Calculate global thresholds (P95) for each 'component-KPI' time series
global_thresholds = metrics_df.groupby('component_KPI')['value'].quantile(0.95).reset_index()
global_thresholds.columns = ['component_KPI', 'P95']

metrics_df, global_thresholds

```
Out[2]:
```
The `metrics.parquet` file was successfully loaded, and the metrics were aggregated by 'component-KPI' (a combination of component name and KPI). Global thresholds (P95) were calculated for each 'component-KPI' time series. This data will help identify anomalies in the next step by comparing actual values against these thresholds. The dataset contains 83,920 rows of metrics and 306 unique 'component-KPI' combinations with their respective P95 thresholds.

The original code execution output of IPython Kernel is also provided below for reference:

(                   time domain    entity_set                         entity_id entity_name                 metric     value                                     metric_set_id service           component_KPI
0      1774697256000000    k8s      k8s.node                                                node_not_ready_status  0.000000  k8s@metric_set@k8s.metric.high_level_metric_node    None  -node_not_ready_status
1      1774697286000000    k8s      k8s.node                                                node_not_ready_status  0.000000  k8s@metric_set@k8s.metric.high_level_metric_node    None  -node_not_ready_status
2      1774697316000000    k8s      k8s.node                                                node_not_ready_status  0.000000  k8s@metric_set@k8s.metric.high_level_metric_node    None  -node_not_ready_status
3      1774697346000000    k8s      k8s.node                                                node_not_ready_status  0.000000  k8s@metric_set@k8s.metric.high_level_metric_node    None  -node_not_ready_status
4      1774697376000000    k8s      k8s.node                                                node_not_ready_status  0.000000  k8s@metric_set@k8s.metric.high_level_metric_node    None  -node_not_ready_status
...                 ...    ...           ...                               ...         ...                    ...       ...                                               ...     ...                     ...
83915  1774701036000000    apm  apm.instance  7d6d334fd534e7f6e8e70d70849df048   inventory             error_rate  0.991554            apm@metric_set@apm.metric.apm.instance    None    inventory-error_rate
83916  1774701066000000    apm  apm.instance  7d6d334fd534e7f6e8e70d70849df048   inventory             error_rate  0.991567            apm@metric_set@apm.metric.apm.instance    None    inventory-error_rate
83917  1774701096000000    apm  apm.instance  7d6d334fd534e7f6e8e70d70849df048   inventory             error_rate  0.990891            apm@metric_set@apm.metric.apm.instance    None    inventory-error_rate
83918  1774701126000000    apm  apm.instance  7d6d334fd534e7f6e8e70d70849df048   inventory             error_rate  0.991758            apm@metric_set@apm.metric.apm.instance    None    inventory-error_rate
83919  1774701156000000    apm  apm.instance  7d6d334fd534e7f6e8e70d70849df048   inventory             error_rate  0.991142            apm@metric_set@apm.metric.apm.instance    None    inventory-error_rate

[83920 rows x 10 columns],                                          component_KPI           P95
0                    -apiserver_concurrent_connections      6.000000
1                  -apiserver_non_2xx_read_request_qps      0.160214
2                 -apiserver_non_2xx_write_request_qps      0.107263
3                       -apiserver_requests_per_second     26.800510
4                           -cluster_failed_node_count      0.000000
..                                                 ...           ...
301                                  shipping-workload  13550.000000
302  shipping::oteldemo.ShippingService/GetQuote-av...      0.003528
303  shipping::oteldemo.ShippingService/GetQuote-re...   6771.500000
304  shipping::oteldemo.ShippingService/ShipOrder-a...      0.000014
305  shipping::oteldemo.ShippingService/ShipOrder-r...   6778.500000

[306 rows x 2 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pytz
from datetime import datetime

# Convert alert start and end times to Unix microseconds for filtering
timezone = pytz.timezone('Asia/Shanghai')
alert_start_us = int(datetime.fromisoformat(alert_start).astimezone(timezone).timestamp() * 1_000_000)
alert_end_us = int(datetime.fromisoformat(alert_end).astimezone(timezone).timestamp() * 1_000_000)

# Filter metrics data within the alert time window
filtered_metrics = metrics_df[(metrics_df['time'] >= alert_start_us) & (metrics_df['time'] <= alert_end_us)]

# Merge with global thresholds to identify anomalies
filtered_metrics = filtered_metrics.merge(global_thresholds, on='component_KPI', how='left')

# Mark anomalies where the value exceeds the P95 threshold
filtered_metrics['is_anomaly'] = filtered_metrics['value'] > filtered_metrics['P95']

# Filter only the anomalies
anomalies = filtered_metrics[filtered_metrics['is_anomaly']]

filtered_metrics, anomalies

```
Out[3]:
```
The metrics data was filtered to include only the alert time window ('2026-03-28T20:27:36.369643+08:00' to '2026-03-28T20:31:01+08:00'). Anomalies were identified by comparing the values of each 'component-KPI' time series against their respective P95 thresholds. Out of 4,013 data points in the filtered dataset, 158 were marked as anomalies where the values exceeded the thresholds. These anomalies will help pinpoint the components and metrics that may have contributed to the issue.

The original code execution output of IPython Kernel is also provided below for reference:

(                  time domain    entity_set                         entity_id entity_name  ...                                     metric_set_id  service           component_KPI       P95 is_anomaly
0     1774700886000000    k8s      k8s.node                                                ...  k8s@metric_set@k8s.metric.high_level_metric_node     None  -node_not_ready_status  0.000000      False
1     1774700916000000    k8s      k8s.node                                                ...  k8s@metric_set@k8s.metric.high_level_metric_node     None  -node_not_ready_status  0.000000      False
2     1774700946000000    k8s      k8s.node                                                ...  k8s@metric_set@k8s.metric.high_level_metric_node     None  -node_not_ready_status  0.000000      False
3     1774700976000000    k8s      k8s.node                                                ...  k8s@metric_set@k8s.metric.high_level_metric_node     None  -node_not_ready_status  0.000000      False
4     1774701006000000    k8s      k8s.node                                                ...  k8s@metric_set@k8s.metric.high_level_metric_node     None  -node_not_ready_status  0.000000      False
...                ...    ...           ...                               ...         ...  ...                                               ...      ...                     ...       ...        ...
4008  1774700916000000    apm  apm.instance  7d6d334fd534e7f6e8e70d70849df048   inventory  ...            apm@metric_set@apm.metric.apm.instance     None    inventory-error_rate  0.992064      False
4009  1774700946000000    apm  apm.instance  7d6d334fd534e7f6e8e70d70849df048   inventory  ...            apm@metric_set@apm.metric.apm.instance     None    inventory-error_rate  0.992064      False
4010  1774700976000000    apm  apm.instance  7d6d334fd534e7f6e8e70d70849df048   inventory  ...            apm@metric_set@apm.metric.apm.instance     None    inventory-error_rate  0.992064      False
4011  1774701006000000    apm  apm.instance  7d6d334fd534e7f6e8e70d70849df048   inventory  ...            apm@metric_set@apm.metric.apm.instance     None    inventory-error_rate  0.992064      False
4012  1774701036000000    apm  apm.instance  7d6d334fd534e7f6e8e70d70849df048   inventory  ...            apm@metric_set@apm.metric.apm.instance     None    inventory-error_rate  0.992064      False

[4013 rows x 12 columns],                   time domain    entity_set                         entity_id                      entity_name  ...                                     metric_set_id  service                               component_KPI        P95 is_anomaly
276   1774700886000000    k8s      k8s.node                                                                     ...  k8s@metric_set@k8s.metric.high_level_metric_node     None                       -node_disk_usage_rate  33.007048       True
277   1774700916000000    k8s      k8s.node                                                                     ...  k8s@metric_set@k8s.metric.high_level_metric_node     None                       -node_disk_usage_rate  33.007048       True
278   1774700946000000    k8s      k8s.node                                                                     ...  k8s@metric_set@k8s.metric.high_level_metric_node     None                       -node_disk_usage_rate  33.007048       True
279   1774700976000000    k8s      k8s.node                                                                     ...  k8s@metric_set@k8s.metric.high_level_metric_node     None                       -node_disk_usage_rate  33.007048       True
280   1774701006000000    k8s      k8s.node                                                                     ...  k8s@metric_set@k8s.metric.high_level_metric_node     None                       -node_disk_usage_rate  33.007048       True
...                ...    ...           ...                               ...                              ...  ...                                               ...      ...                                         ...        ...        ...
3982  1774701036000000    apm  apm.instance  fa548039d9a7b76db9ec70507e734faf  frontend-proxy-67d64dc4f8-8c99b  ...            apm@metric_set@apm.metric.apm.instance     None  frontend-proxy-67d64dc4f8-8c99b-error_rate   0.208525       True
3988  1774701036000000    apm  apm.instance  d1f1b0349fadcfdc23ca3918ac448623  frontend-proxy-67d64dc4f8-xz2fd  ...            apm@metric_set@apm.metric.apm.instance     None  frontend-proxy-67d64dc4f8-xz2fd-error_rate   0.203603       True
3999  1774701006000000    apm  apm.instance  edd1b3a198ca77c10c1e4414044e4671            cart-67f957cc88-njsn7  ...            apm@metric_set@apm.metric.apm.instance     None            cart-67f957cc88-njsn7-error_rate   0.880373       True
4000  1774701036000000    apm  apm.instance  edd1b3a198ca77c10c1e4414044e4671            cart-67f957cc88-njsn7  ...            apm@metric_set@apm.metric.apm.instance     None            cart-67f957cc88-njsn7-error_rate   0.880373       True
4006  1774701036000000    apm  apm.instance  82ae8fb5979d5759841affc9d937a0d4        frontend-6df7d557f5-nbvwh  ...            apm@metric_set@apm.metric.apm.instance     None        frontend-6df7d557f5-nbvwh-error_rate   0.211925       True

[158 rows x 12 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Add a column to calculate the percentage of threshold breach
anomalies['threshold_breach'] = (anomalies['value'] - anomalies['P95']) / anomalies['P95']

# Filter anomalies where the threshold breach is substantial (>50% of the threshold)
significant_anomalies = anomalies[anomalies['threshold_breach'] > 0.5]

# Group anomalies by 'component-KPI' and identify consecutive anomalies
# Sort by 'component-KPI' and 'time' to ensure proper grouping
significant_anomalies = significant_anomalies.sort_values(by=['component_KPI', 'time'])

# Add a column to identify consecutive anomalies
significant_anomalies['consecutive_group'] = (
    (significant_anomalies['time'].diff() > 1_000_000 * 60) |  # 1-minute gap in microseconds
    (significant_anomalies['component_KPI'] != significant_anomalies['component_KPI'].shift())
).cumsum()

# Group by 'component-KPI' and 'consecutive_group' to focus on significant deviations
grouped_anomalies = significant_anomalies.groupby(['component_KPI', 'consecutive_group'])

# Aggregate the results to summarize the significant deviations
fault_summary = grouped_anomalies.agg(
    start_time=('time', 'min'),
    end_time=('time', 'max'),
    max_breach=('threshold_breach', 'max'),
    anomaly_count=('time', 'count')
).reset_index()

# Filter out groups with only isolated noise (e.g., single anomaly in a group)
fault_summary = fault_summary[fault_summary['anomaly_count'] > 1]

fault_summary

```
Out[4]:
```
Two significant fault groups were identified during the alert time window:

1. The `-deployment_cpu_usage_total` metric experienced consecutive anomalies from `1774700886000000` to `1774701036000000` (UTC microseconds), with a maximum threshold breach of 130.96% and 6 anomalies in total.

2. The `-deployment_cpu_usage_vs_limits` metric experienced consecutive anomalies from `1774700886000000` to `1774701036000000` (UTC microseconds), with a maximum threshold breach of 120.98% and 6 anomalies in total.

These metrics indicate substantial deviations in CPU usage, suggesting potential resource-related issues.

The original code execution output of IPython Kernel is also provided below for reference:

component_KPI  consecutive_group        start_time          end_time  max_breach  anomaly_count
1      -deployment_cpu_usage_total                  2  1774700886000000  1774701036000000    1.309557              6
2  -deployment_cpu_usage_vs_limits                  3  1774700886000000  1774701036000000    1.209769              6```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_path = f'{task_directory}/traces.parquet'
traces_df = pd.read_parquet(traces_path)

# Ensure 'startTime', 'endTime', and 'duration' columns are numeric
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')
traces_df['duration'] = pd.to_numeric(traces_df['duration'], errors='coerce')

# Filter traces within the alert time window
filtered_traces = traces_df[
    (traces_df['startTime'] >= alert_start_us * 1_000) & 
    (traces_df['endTime'] <= alert_end_us * 1_000)
]

# Identify traces with errors (statusCode != 0 or statusMessage indicating an error)
error_traces = filtered_traces[
    (filtered_traces['statusCode'] != 0) | 
    (filtered_traces['statusMessage'].str.contains('error', case=False, na=False))
]

# Analyze call chains to determine the most downstream faulty service
# Sort by 'startTime' to ensure proper ordering
error_traces = error_traces.sort_values(by='startTime')

# Group by 'traceId' to analyze call chains
call_chain_analysis = error_traces.groupby('traceId').agg(
    most_downstream_service=('serviceName', 'last'),
    error_count=('spanId', 'count')
).reset_index()

call_chain_analysis

```
Out[5]:
```
The analysis of traces within the alert time window identified 9,152 traces with errors. Each trace was analyzed to determine the most downstream faulty service. Examples of services with errors include `cart` (e.g., 18 errors in one trace), `accounting` (e.g., 1 error in one trace), and `product-catalog` (e.g., 35 errors in one trace). These results highlight multiple services experiencing errors, with `cart` and `product-catalog` showing significant error counts in some traces. Further investigation is needed to pinpoint the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

traceId most_downstream_service  error_count
0     00034a1076635bf231839cd12cb3e52b                    cart           18
1     0004246488a5db7c492a4a4e5a687269              accounting            1
2     000676aeff3e2cc63f90f3653d6a9b5e         product-catalog            7
3     00131aa54f6c81c50bb47d6b3e6abd4d         product-catalog           35
4     001c7246362e0254843e2b14422fa241                    cart           10
...                                ...                     ...          ...
9147  ffee815b7ebca76a43f6efa9d99b8b74                   email           29
9148  ffeec975da460196393d544cc7dc2c75               inventory           10
9149  fff27b14eb8e98b8e19c76efd23227ea              accounting            1
9150  fff302b93c094d612e8ee85c816b74dc                currency           12
9151  fff54bc2bf43f4f8b2eee313b06bb958         product-catalog            7

[9152 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_path = f'{task_directory}/logs.parquet'
logs_df = pd.read_parquet(logs_path)

# Ensure '_time_' column is in numeric format for filtering
logs_df['_time_'] = pd.to_numeric(logs_df['_time_'], errors='coerce')

# Filter logs within the alert time window
filtered_logs = logs_df[
    (logs_df['_time_'] >= alert_start_us) & 
    (logs_df['_time_'] <= alert_end_us)
]

# Search for error-related logs for the `inventory` service and its upstream services
# Services of interest: inventory, cart, product-catalog
services_of_interest = ['inventory', 'cart', 'product-catalog']
error_logs = filtered_logs[
    (filtered_logs['_source_'].str.contains('|'.join(services_of_interest), case=False, na=False)) &
    (filtered_logs['content'].str.contains('error|exception|fail|critical', case=False, na=False))
]

error_logs

```
Out[6]:
```
No error-related logs were found for the `inventory` service or its upstream services (`cart`, `product-catalog`, etc.) within the alert time window. This suggests that the root cause may not be directly observable in the application logs or may require further investigation in other telemetry data sources.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```